In [ ]:
from nyc_taxi_trips_etl.utils.pipeline_params_parsing import (
    parse_and_expand_year_months,
)

year_months_to_extract = dbutils.widgets.get("year_months_to_extract")
catalog = dbutils.widgets.get("catalog")

In [ ]:
year_months = parse_and_expand_year_months(year_months_to_extract)

In [ ]:
peak_hours_breakdown_df = spark.sql(f"""
    SELECT
        year_month,
        pickup_hour,
        COUNT(*) AS trip_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS trip_percentage
    FROM {catalog}.silver.yellow_trip_data
    WHERE year_month IN ({",".join([f"'{ym}'" for ym in year_months])})
    GROUP BY year_month, pickup_hour
    ORDER BY year_month, trip_count DESC
""")

In [ ]:
(
    peak_hours_breakdown_df.write.mode("overwrite")
    .option(
        "replaceWhere",
        f"""year_month in ({",".join([f"'{ym}'" for ym in year_months])})""",
    )
    .saveAsTable(f"{catalog}.gold.trip_peak_hours")
)